# IMPORTS

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

warnings.filterwarnings("ignore")

# setting the seed
RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
TARGET_COL = "House"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# LOAD DATA

In [6]:
characters_path = f"{DATA_DIR}/Characters.csv"
df = pd.read_csv(characters_path, delimiter=';')

print("Shape:", df.shape)
df.head()

Shape: (140, 15)


,Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death
0,1,Harry James Potter,Male,Student,Gryffindor,"11"" Holly phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore's Army | Order o...,Parseltongue| Defence Against the Dark Arts | ...,31 July 1980,NaN
1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Wizard chess | Quidditch goalkeeping,1 March 1980,NaN
2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾"" vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore's Army | Order of the Phoenix | Hog...,Almost everything,"19 September, 1979",NaN
3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,"15"" Elder Thestral tail hair core",Phoenix,Human,Half-blood,Silver| formerly auburn,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,"30 June, 1997"
4,5,Rubeus Hagrid,Male,Keeper of Keys and Grounds | Professor of Care...,Gryffindor,"16"" Oak unknown core",NaN,Half-Human/Half-Giant,Part-Human (Half-giant),Black,Black,Albus Dumbledore | Order of the Phoenix | Hogw...,Resistant to stunning spells| above average st...,6 December 1928,NaN


In [9]:
df = df.dropna(subset=[TARGET_COL])
print(f"Shape después de eliminar NaN en {TARGET_COL}:", df.shape)

Shape después de eliminar NaN en House: (101, 15)


In [11]:
counts = df[TARGET_COL].value_counts()
df = df[df[TARGET_COL].isin(counts[counts > 1].index)]
print(f"Shape después de eliminar clases poco frecuentes en {TARGET_COL}:", df.shape)

Shape después de eliminar clases poco frecuentes en House: (100, 15)


In [15]:
df.head().to_csv()

',Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death\r\n0,1,Harry James Potter,Male,Student,Gryffindor,"11""  Holly  phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Parseltongue| Defence Against the Dark Arts | Seeker,31 July 1980,\r\n1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair ",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Wizard chess | Quidditch goalkeeping,1 March 1980,\r\n2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾""  vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore\'s Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry,Almost everything,"19 September,\xa01979",\r\n3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gry

In [16]:
spells_path = f"{DATA_DIR}/Spells.csv"
df_spells = pd.read_csv(spells_path, delimiter=';')

print("Shape:", df_spells.shape)
df_spells.head().to_csv()

Shape: (301, 5)


',Name,Incantation,Type,Effect,Light\r\n0,Summoning Charm,Accio,Charm,Summons an object,\r\n1,Age Line,Unknown,Charm,Prevents people above or below a certain age from access to a target,Blue\r\n2,Water-Making Spell,Aguamenti,"Charm, Conjuration",Conjures\xa0water,Icy blue\r\n3,Launch an object up into the air,Alarte Ascendare,Charm,Rockets target upward,Red\r\n4,Albus Dumbledore\'s Forceful Spell,Unknown,Spell,Great Force,\r\n'

In [18]:
hp1_path = f"{DATA_DIR}/Harry Potter 1.csv"
df_hp1 = pd.read_csv(hp1_path, delimiter=';')

print("Shape:", df_hp1.shape)
df_hp1.head().to_csv()

Shape: (1700, 2)


',Character,Sentence\r\n0,HARRY ,"I can’t let you out, Hedwig. "\r\n1,HARRY ,I’m not allowed to use magic outside of school. \r\n2,HARRY ,"Besides, if Uncle Vernon…"\r\n3,VERNON,Harry Potter!\r\n4,HARRY,Now you’ve done it.\r\n'

In [20]:
hp2_path = f"{DATA_DIR}/Harry Potter 2.csv"
df_hp2 = pd.read_csv(hp2_path, delimiter=';')

print("Shape:", df_hp2.shape)
df_hp2.head().to_csv()

Shape: (1700, 2)


',Character,Sentence\r\n0,HARRY ,"I can’t let you out, Hedwig. "\r\n1,HARRY ,I’m not allowed to use magic outside of school. \r\n2,HARRY ,"Besides, if Uncle Vernon…"\r\n3,VERNON,Harry Potter!\r\n4,HARRY,Now you’ve done it.\r\n'

In [19]:
hp3_path = f"{DATA_DIR}/Harry Potter 3.csv"
df_hp3 = pd.read_csv(hp3_path, delimiter=';')

print("Shape:", df_hp3.shape)
df_hp3.head().to_csv()

Shape: (1638, 2)


',CHARACTER,SENTENCE\r\n0,HARRY,Lumos Maxima...\r\n1,HARRY,Lumos Maxima...\r\n2,HARRY,Lumos Maxima...\r\n3,HARRY,Lumos... MAXIMA!\r\n4,AUNT PETUNIA,Harry! Harry!\r\n'

In [13]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (80, 14) Test shape: (20, 14)


# FEATURE ENGINEERING FUNCTIONS

In [ ]:
def normalize_character_name(name: str) -> str:
    if pd.isna(name):
        return 'unknown'
    name = name.lower().strip()
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'[^a-z\s]', '', name)
    return name

def load_and_merge_quotes(csv_list):
    dfs = [pd.read_csv(path, delimiter=';') for path in csv_list]
    return pd.concat(dfs, ignore_index=True)

def clean_quotes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Character'] = df['Character'].apply(normalize_character_name)
    df['Sentence'] = df['Sentence'].astype(str).str.lower().str.strip()
    df['Sentence'] = df['Sentence'].str.replace('"', '', regex=False)
    df = df[df['Sentence'].str.len() > 0]
    return df

def build_quotes_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['sentence_length'] = df['Sentence'].apply(len)
    df['word_count'] = df['Sentence'].apply(lambda x: len(x.split()))
    df['question'] = df['Sentence'].str.contains(r'\?').astype(int)
    df['exclamation'] = df['Sentence'].str.contains(r'!').astype(int)
    agg = df.groupby('Character').agg({
        'sentence_length': 'mean',
        'word_count': 'mean',
        'question': 'mean',
        'exclamation': 'mean'
    })
    counts = df.groupby('Character').size().to_frame('num_quotes')
    return agg.join(counts).reset_index()

def clean_characters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_cols = ['Id', 'Death', 'Birth', 'Loyalty', 'Skills', 'Wand', 'Job']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.lower().str.strip()
    df = df.fillna('unknown')
    return df

def merge_characters_quotes(char_df: pd.DataFrame, quotes_feat: pd.DataFrame):
    df = char_df.copy()
    # Normalizar nombres en ambos dataframes usando la misma función
    df['char_key'] = df['Name'].apply(normalize_character_name)
    quotes_feat['char_key'] = quotes_feat['Character'].apply(normalize_character_name)
    merged = df.merge(quotes_feat, on='char_key', how='left')
    return merged.fillna(0)

def spells_keyword_features(df: pd.DataFrame, spells_df: pd.DataFrame):
    df = df.copy()
    spells = spells_df['Incantation'].dropna().str.lower().tolist()
    def count_spells(text):
        return sum(spell in text for spell in spells)
    df['spell_mentions'] = df['Sentence'].apply(count_spells)
    spell_usage = df.groupby('Character')['spell_mentions'].sum().reset_index()
    return spell_usage

In [ ]:
quotes_df = load_and_merge_quotes([
    "../data/Harry Potter 1.csv",
    "../data/Harry Potter 2.csv",
    "../data/Harry Potter 3.csv"
])

In [ ]:
# --- Feature engineering y asignación de variables finales ---
# Limpiar y preparar quotes
quotes_clean = clean_quotes(quotes_df)
# Limpiar personajes
X_train_clean = clean_characters(X_train)
X_test_clean = clean_characters(X_test)
# Calcular features de quotes
quotes_features = build_quotes_features(quotes_clean)
# Hacer el merge usando la normalización correcta
X_train_fe = merge_characters_quotes(X_train_clean, quotes_features)
X_test_fe = merge_characters_quotes(X_test_clean, quotes_features)
# Mostrar resultados
print("X_train_fe:")
display(X_train_fe.head())
print("X_test_fe:")
display(X_test_fe.head())

X_train_fe:


,Name,Gender,Patronus,Species,Blood status,Hair colour,Eye colour,char_key,Character,sentence_length,word_count,question,exclamation,num_quotes
0,alecto carrow,female,unknown,human,pure-blood or half-blood,unknown,unknown,alecto carrow,0,0.000000,0.000000,0.000000,0.000000,0.0
1,gilderoy lockhart,male,non-corporeal,human,half-blood,blond,blue,gilderoy lockhart,gilderoy lockhart,36.318584,6.725664,0.150442,0.097345,113.0
2,parvati patil,female,unknown,human,pure-blood or half-blood,dark,dark,parvati patil,0,0.000000,0.000000,0.000000,0.000000,0.0
3,salazar slytherin,male,unknown,human,pure-blood,grey,dark,salazar slytherin,0,0.000000,0.000000,0.000000,0.000000,0.0
4,lucius malfoy,male,unknown,human,pure-blood,white-blond,grey,lucius malfoy,lucius malfoy,33.973333,6.453333,0.186667,0.026667,75.0


X_test_fe:


,Name,Gender,Patronus,Species,Blood status,Hair colour,Eye colour,char_key,Character,sentence_length,word_count,question,exclamation,num_quotes
0,cho chang,female,swan,human,pure-blood or half-blood,black,dark,cho chang,0,0.0,0.0,0.0,0.0,0.0
1,roger davies,male,unknown,human,unknown,dark,dark,roger davies,0,0.0,0.0,0.0,0.0,0.0
2,corban yaxley,male,unknown,human,pure-blood or half-blood,blonde,blue,corban yaxley,0,0.0,0.0,0.0,0.0,0.0
3,regulus arcturus black,male,non-corporeal,human,pure-blood,black,unknown,regulus arcturus black,0,0.0,0.0,0.0,0.0,0.0
4,newton scamander,male,unknown,human,pure-blood or half-blood,red brown,blue,newton scamander,0,0.0,0.0,0.0,0.0,0.0
5,garrick ollivander,male,non-corporeal,human,half-blood,unknown,silvery,garrick ollivander,0,0.0,0.0,0.0,0.0,0.0
6,katie bell,female,unknown,human,pure-blood or half-blood,brown,brown,katie bell,0,0.0,0.0,0.0,0.0,0.0
7,arthur weasley,male,weasel,human,pure-blood,red,green,arthur weasley,0,0.0,0.0,0.0,0.0,0.0
8,draco malfoy,male,unknown,human,pure-blood,white-blond,grey,draco malfoy,0,0.0,0.0,0.0,0.0,0.0
9,gabrielle delacour,female,unknown,human,quarter-veela,silvery-blonde,unknown,gabrielle delacour,0,0.0,0.0,0.0,0.0,0.0
